## Initialization

In [ ]:
# Imports
# import pickle
# from pathlib import Path
from typing import Callable, Literal
from random import sample
from math import ceil

import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
from scipy.signal import find_peaks
from scipy.interpolate import CubicSpline
import pandas as pd
# from scipy.interpolate import interp1d
# from data_processing.processing.bimodal_fitting import (
#     get_psd_energy_histogram,
#     scan_histogram_slices,
#     find_failed_slices,
#     BimodalBounds,
#     BimodalParams
# )
# from data_processing.processing.calibration import Detector, recalibrate
# from data_processing.processing.figure_of_merit import gaussian
# from data_processing.processing.neutron_classification import classify
# from data_processing.processing.neutron_window_generation import (
#     generate_nasa_neutron_window,
#     generate_n_distro_neutron_window
# )
from data_processing import processing as proc
from data_processing import loading as load
from data_processing import types as proc_types
# from data_processing.arc_paths import (INPUT_DATA_FOLDER, get_exp_root,
#                                        get_parq_root)
from data_processing.dataframe_validation import DetectorDataframeColumn, EnergyColumn
from data_processing.experiment_data_keys import (ExperimentDataKey,
                                                  ExperimentNeutronData)
from data_processing.helpers import (
    get_input_with_default,
    # get_input_required,
    input_experiment_ids,
    stop
)
# from data_processing.loading import get_neutron_window_paths, load_side_borders
# from data_processing.loading.dataframe_loading import load_psd
# from data_processing.loading.timetag_processing import calculate_timetag_hours
# from data_processing.reporting import plot_classification
# from data_processing.types import (BimodalBounds, BimodalParams,
#                                    NasaGenerationSettings,
#                                    NeutronWindowSettings, WindowType)
# from scipy.optimize import curve_fit
# from scipy.signal import deconvolve

In [ ]:
CalibrationKey = Literal[ExperimentDataKey.CAEN_CALIBRATION, ExperimentDataKey.NEW_CALIBRATION]
NasaBorderKey = Literal[ExperimentDataKey.NASA_BORDERS, ExperimentDataKey.NASA_BORDERS_RECALC]

### Functions

In [ ]:
def get_nasa_generation_settings(
    calib_key: CalibrationKey
) -> proc_types.NasaGenerationSettings:
    sigma = get_input_with_default(
        """\
Enter value of sigma
Press Enter for default (5)
""",
        5,
        float
    )
    window_offset = get_input_with_default(
        """\
Enter value of offset between top and bottom window border
Press Enter for default (0.2)
""",
        0.2,
        float
    )
    left_border_type_input = get_input_with_default(
        """\
How do you want to handle the left border?
1: use existing value (default)
2: recalculate from data
3: enter own value
Press Enter for default
""",
        1,
        int
    )
    if left_border_type_input == 1:
        existing_left_border_version_input = get_input_with_default(
            """\
Which existing left border do you want to use?
1: original (0.1966 MeVee) (default)
2: newer (~0.1866 MeVee)
or press Enter for default
""",
            1,
            int
        )
        if existing_left_border_version_input in [1, 2]:
            border_key = ExperimentDataKey.NASA_BORDERS if existing_left_border_version_input == 1 else ExperimentDataKey.NASA_BORDERS_RECALC
            file_name_prefix = f"{calib_key.value}_{border_key.value}"
            side_borders_path, *_ = load.get_neutron_window_paths(file_name_prefix=file_name_prefix)
            left_border, _ = load.load_side_borders(side_borders_path=side_borders_path)
            if left_border is None:
                raise ValueError("Left border could not be loaded")
            lower_energy_bound = left_border
            recalc_lower_bound = False
        else:
            raise ValueError("Unsupported choice")
        pass
    elif left_border_type_input == 2:
        lower_energy_bound = 0.1966
        recalc_lower_bound = True
    elif left_border_type_input == 3:
        lower_energy_bound = get_input_with_default(
            """\
Enter value of lower energy bound (in MeVee)
Press Enter for default (0.1966)
""",
            0.1966,
            float
        )
        recalc_lower_bound = False
    else:
        raise ValueError("Unsupported choice")
    settings = proc_types.NasaGenerationSettings(
        window_offset=window_offset,
        sigma=sigma,
        lower_energy_bound=lower_energy_bound,
        recalculate_lower_energy_bound=recalc_lower_bound
    )
    return settings


def make_strategy_factory_fn(
    strategy_factory: proc.NeutronStrategyFactory,
    window_type: proc_types.WindowType,
    loading: bool,
    settings: proc_types.NeutronWindowSettings
) -> Callable[[], proc.AbstractNeutronStrategy]:
    def factory_fn():
        return strategy_factory.make_neutron_window_strategy(
            window_type, loading, settings
        )
    return factory_fn


def make_strategy_for_experiments(
    experiment_neutron_data: ExperimentNeutronData, 
    factory_fn: Callable[[], proc.AbstractNeutronStrategy]
) -> ExperimentNeutronData:
    new_neutron_data = {
        exp_id: {**exp_data, ExperimentDataKey.BORDER_STRATEGY: factory_fn()}
        for exp_id, exp_data
        in experiment_neutron_data.items()
    }
    return new_neutron_data

In [ ]:
def moving_average(arr, n=5):
    ret = np.cumsum(arr, dtype=float)
    ret[n:] = ret[n:] - ret[:-n]
    mov_avg = ret[n-1:] / n
    prefix = np.empty((n-1,))
    prefix[:] = np.nan
    return np.concatenate((prefix, mov_avg))


def moving_average_centered(arr, n=5):
    if n % 2 != 1:
        raise ValueError("Centered moving average needs odd window size")
    prefix_count = (n-1)//2
    ret = np.nancumsum(arr, dtype=float)
    ret[n:] = ret[n:] - ret[:-n]
    mov_avg = ret[n-1:] / n
    prefix = np.empty((prefix_count,))
    suffix = np.empty((prefix_count,))
    prefix[:] = np.nan
    suffix[:] = np.nan
    return np.concatenate((prefix, mov_avg, suffix))

## Experiment ID Input

In [ ]:
# experiment_ids = input_experiment_ids()
# experiment_ids = ["TB-unmatched", "TB-47"]
# experiment_ids = ["TB-26", "TB-matched"]
# experiment_ids = ["TB-25", "TB-26", "TB-27", "TB-30"]
experiment_ids = ["TB-26"]

In [ ]:
# more here?
experiment_neutron_data: ExperimentNeutronData = {
    exp_id: {}
    for exp_id in experiment_ids
}

In [ ]:
# calib_input = get_input_with_default(
#     "Do you want to use new calibration? [y/n, or press Enter for yes]",
#     "y",
#     str
# )
calib_input = "y"

is_new_calibration = calib_input.lower() == "y"
calibrated_energy_column: EnergyColumn = (
    DetectorDataframeColumn.RECALIBRATED_ENERGY
    if is_new_calibration else DetectorDataframeColumn.CALIB_ENERGY
)
calib_key: CalibrationKey = ExperimentDataKey.NEW_CALIBRATION if is_new_calibration else ExperimentDataKey.CAEN_CALIBRATION

In [ ]:
strategy_factory = proc.NeutronStrategyFactory()
# settings = get_nasa_generation_settings(calib_key)
window_offset = 0.2
sigma = 5
lower_energy_bound = 0.1966
recalc_lower_bound = False
settings = proc_types.NasaGenerationSettings(
        window_offset=window_offset,
        sigma=sigma,
        lower_energy_bound=lower_energy_bound,
        recalculate_lower_energy_bound=recalc_lower_bound
    )
factory_fn = make_strategy_factory_fn(
    strategy_factory, "nasa", False, settings)
experiment_neutron_data = make_strategy_for_experiments(
    experiment_neutron_data, factory_fn)

## Data Loading and Initial Processing

In [ ]:
# Data Loading
for exp_id, exp_data in experiment_neutron_data.items():
    # exp_data[ExperimentDataKey.UNCLASSIFIED] = load.load_parquet_psd(exp_id, with_flags=True)
    # exp_data["signals_df"] = load.load_parquet_signals(exp_id)
    exp_data[ExperimentDataKey.UNCLASSIFIED] = load.load_caen_csvs(exp_id, get_flags=True, raw=True)

In [ ]:
# Data Loading
for exp_id, exp_data in experiment_neutron_data.items():
    exp_data["signals_df"] = load.load_caen_csvs(exp_id, get_psd=False, get_signals=True, raw=True)

In [ ]:
# Express timetags in hours elapsed
for exp_id, exp_data in experiment_neutron_data.items():
    unclassified_df = exp_data[ExperimentDataKey.UNCLASSIFIED]
    unclassified_df = load.calculate_timetag_hours(unclassified_df)
    exp_data[ExperimentDataKey.UNCLASSIFIED] = unclassified_df

In [ ]:
# Recalibrate energy
for exp_id, exp_data in experiment_neutron_data.items():
    unclassified_df = exp_data[ExperimentDataKey.UNCLASSIFIED]
    unclassified_df = proc.recalibrate(unclassified_df, proc.Detector.ZERO)
    exp_data[ExperimentDataKey.UNCLASSIFIED] = unclassified_df

## Neutron Classification

In [ ]:
# Generate histogram

start_scan_idx = 0
end_scan_idx = 420
energy_width = 20e-3

for exp_id, exp_data in experiment_neutron_data.items():
    psd_report = exp_data[ExperimentDataKey.UNCLASSIFIED]
    Z, xe, ye = proc.get_psd_energy_histogram(
        psd_report,
        calibrated_energy_column,
        energy_width=energy_width
    )
    exp_data[ExperimentDataKey.PSD_HISTOGRAM] = Z
    exp_data[ExperimentDataKey.HISTOGRAM_X_EDGES] = xe
    exp_data[ExperimentDataKey.HISTOGRAM_Y_EDGES] = ye
    exp_data[ExperimentDataKey.END_SCAN_IDX] = min(end_scan_idx, len(Z))

In [ ]:
# TODO get fit dataframe (not needed if loading, but do anyway to keep process consistent)
stop_here = False

for exp_id, exp_data in experiment_neutron_data.items():
    Z = exp_data[ExperimentDataKey.PSD_HISTOGRAM]
    xe = exp_data[ExperimentDataKey.HISTOGRAM_X_EDGES]
    ye = exp_data[ExperimentDataKey.HISTOGRAM_Y_EDGES]
    end_scan_idx = exp_data[ExperimentDataKey.END_SCAN_IDX]

    # # Default
    # default_bounds: BimodalBounds = (
    #     BimodalParams(0.1, 0.01, 1, 0.25, 0.01, 0),
    #     BimodalParams(0.2, 0.1, Z.max(), 0.38, 0.04, 4000)
    # )

    # bounds_a: BimodalBounds = (
    #     BimodalParams(0.1, 0.01, 1, 0.34, 0.01, 0),
    #     BimodalParams(0.2, 0.1, Z.max(), 0.36, 0.04, 4000)
    # )

    # bounds_b: BimodalBounds = (
    #     BimodalParams(0.1, 0.01, 1, 0.34, 0.01, 0),
    #     BimodalParams(0.2, 0.1, Z.max(), 0.36, 0.03, 4000)
    # )

    # # Ranged Example
    # bounds = [
    #     ((0, 60), bounds_a),
    # ]

    df, df_err = proc.scan_histogram_slices(
        Z,
        xe,
        ye,
        fit_style="peak_finder",
        # default_bounds,
        # bounds=bounds,
        start_idx=start_scan_idx,
        end_idx=end_scan_idx
    )
    df, bad_slice_indexes = proc.find_failed_slices(df, exp_id)

    if bad_slice_indexes is not None:
        exp_data[ExperimentDataKey.VALID_SLICE_FITS] = df
        exp_data[ExperimentDataKey.BAD_SLICE_INDEXES] = bad_slice_indexes
        stop_here = True
    else:
        # exp_data['fom_results'] = df
        exp_data[ExperimentDataKey.FOM_RESULTS] = df

if stop_here:
    stop()

In [ ]:
# get borders from strategy
for exp_id, exp_data in experiment_neutron_data.items():
    if ExperimentDataKey.FOM_RESULTS not in exp_data:
        print(f"No good fit data on Experiment {exp_id}")
        continue

    fom_results = exp_data[ExperimentDataKey.FOM_RESULTS]
    strategy = exp_data[ExperimentDataKey.BORDER_STRATEGY]

    strategy.set_slice_fit_dataframe(fom_results)
    borders = strategy.get_neutron_window()

    exp_data[ExperimentDataKey.BORDERS] = borders

In [ ]:
# classify neutrons
for exp_id, exp_data in experiment_neutron_data.items():
    psd_report = exp_data[ExperimentDataKey.UNCLASSIFIED].copy()
    borders = exp_data[ExperimentDataKey.BORDERS]

    psd_report = proc.classify(
        psd_report,
        calibrated_energy_column,
        borders,
        DetectorDataframeColumn.NEW_N_CLASS
    )

    exp_data[ExperimentDataKey.PSD_REPORT] = psd_report

## Pulse Selection

In [ ]:
for exp_id, exp_data in experiment_neutron_data.items():
    psd_report = exp_data[ExperimentDataKey.PSD_REPORT]
    n_class_col_name = DetectorDataframeColumn.NEW_N_CLASS.value
    
    gamma_only = psd_report.query(f"~{n_class_col_name}").copy()
    neutrons_only = psd_report.query(n_class_col_name).copy()
    exp_data[ExperimentDataKey.NEUTRONS_ONLY] = neutrons_only
    exp_data[ExperimentDataKey.GAMMA_ONLY] = gamma_only

In [ ]:
for exp_id, exp_data in experiment_neutron_data.items():
    print(exp_id)
    # neutrons_only = exp_data[ExperimentDataKey.NEUTRONS_ONLY]
    # gamma_only = exp_data[ExperimentDataKey.GAMMA_ONLY]
    psd_report = exp_data[ExperimentDataKey.PSD_REPORT]
    signals_df = exp_data["signals_df"]

    # print(neutrons_only.shape)
    print(psd_report.shape)
    
    # neutron_signals = signals_df.loc[neutrons_only.index].astype("int32")
    # gamma_signals = signals_df.loc[gamma_only.index].astype("int32")

    # n_signals_np = neutron_signals.to_numpy()
    # n_baselines = n_signals_np.max(axis=1).reshape(-1, 1)
    # n_baselines = n_signals_np[:, :30].mean(axis=1).reshape(-1, 1)
    # n_signals_np = -n_signals_np + n_baselines
    # print(n_signals_np.max())
    # neutron_signals = pd.DataFrame(n_signals_np, index=neutron_signals.index, columns=neutron_signals.columns)
    # neutrons_only["peak_height"] = neutron_signals.max(axis=1)

    # g_signals_np = gamma_signals.to_numpy()
    # # g_baselines = g_signals_np.max(axis=1).reshape(-1, 1)
    # g_baselines = g_signals_np[:, :30].mean(axis=1).reshape(-1, 1)
    # g_signals_np = -g_signals_np + g_baselines
    # print(g_signals_np.max())
    # gamma_signals = pd.DataFrame(g_signals_np, index=gamma_signals.index, columns=gamma_signals.columns)

    signals_np = signals_df.to_numpy()
    baselines = signals_np[:, :30].mean(axis=1).reshape(-1, 1)
    signals_np = -signals_np + baselines
    # print(signals_np.max())
    signals_df = pd.DataFrame(signals_np, index=signals_df.index, columns=signals_df.columns)

    exp_data["signals_df"] = signals_df
    # exp_data["neutron_signals"] = neutron_signals
    # exp_data["gamma_signals"] = gamma_signals

In [ ]:
signals_count = 4000000
for exp_name, exp_data in experiment_neutron_data.items():
    print(exp_name)
    # neutrons_only = exp_data[ExperimentDataKey.NEUTRONS_ONLY]
    # neutron_signals = exp_data["neutron_signals"]
    psd_report = exp_data[ExperimentDataKey.PSD_REPORT]
    signals_df = exp_data["signals_df"]

    # subset = neutrons_only.head(signals_count)
    subset = psd_report.head(signals_count)

    is_pileup = subset[DetectorDataframeColumn.PILEUP.value]
    is_input_saturated = subset[
        DetectorDataframeColumn.INPUT_SATURATING.value]
    is_event_saturated = subset[
        DetectorDataframeColumn.EVENT_OR_TRAP_SATURATING.value]
    is_max_triggers = subset[
        DetectorDataframeColumn.MAX_TRIGGERS_COUNTED.value]

    neutron_pileups = subset[
        is_pileup & (~is_input_saturated) & (~is_event_saturated)
    ]
    neutron_saturated = subset[
        (is_input_saturated | is_event_saturated)
    ]
    # neutron_max_triggers = neutron_subset[
    #     is_max_triggers & (
    #         ~(is_pileup | is_input_saturated | is_event_saturated)
    #     )
    # ]
    neutron_max_triggers = subset[is_max_triggers]
    neutron_clean = subset[
        ~(
            is_pileup |
            is_input_saturated |
            is_event_saturated
        )
    ]

    # subset_signals = signals_df.loc[
    #     subset.index
    # ].astype("int32")
    subset_signals = signals_df.astype("int32")
    pileup_signals = signals_df.loc[
        neutron_pileups.index
    ].astype("int32")
    saturated_signals = signals_df.loc[
        neutron_saturated.index
    ].astype("int32")
    # neutron_max_triggers_signals = neutron_signals.loc[
    #     neutron_max_triggers.index
    # ].astype("int32")
    clean_signals = signals_df.loc[
        neutron_clean.index
    ].astype("int32")

    print("Signal Counts")
    print(f"All:          {subset_signals.shape[0]}")
    print(f"Pileups:      {pileup_signals.shape[0]}")
    print(f"Saturated:    {saturated_signals.shape[0]}")
    # print(f"Max Triggers: {neutron_max_triggers_signals.shape[0]}")
    print(f"Clean:        {clean_signals.shape[0]}")

    exp_data["subset_signals"] = subset_signals
    exp_data["pileup_signals"] = pileup_signals
    exp_data["saturated_signals"] = saturated_signals
    # exp_data["max_trigger_signals"] = neutron_max_triggers_signals
    exp_data["clean_signals"] = clean_signals

In [ ]:
min_peak_height = 500
min_peak_prominence = 50
main_peak_expected_idx = 50
main_peak_window = 14

for exp_name, exp_data in experiment_neutron_data.items():
    clean_signals = exp_data["clean_signals"]

    uncaught_pileup_cols = []
    pileup_peaks = {}

    for i, (row_id, row_data) in enumerate(clean_signals.iterrows()):
        if i % 100000 == 0:
            print(f"{i/1000000:.1f}M", row_id, len(uncaught_pileup_cols))
        row_data.index = row_data.index.map(int)
        peaks, *_ = find_peaks(
            row_data,
            height=min_peak_height,
            prominence=min_peak_prominence
        )
        filtered_peaks = [
            peak for peak in peaks
            if abs(peak - main_peak_expected_idx) > main_peak_window
        ]
        if len(filtered_peaks) > 0:
            peak_heights = [row_data.loc[peak_idx] for peak_idx in filtered_peaks]
            peak_coords = filtered_peaks, peak_heights
            uncaught_pileup_cols.append(row_data)
            pileup_peaks[row_id] = peak_coords
    
    print(len(uncaught_pileup_cols))
    uncaught_pileup_df = pd.concat(uncaught_pileup_cols, axis=1)
    print(uncaught_pileup_df.shape)
    exp_data["uncaught_pileup_df"] = uncaught_pileup_df
    exp_data["pileup_peaks"] = pileup_peaks

## Plotting

In [ ]:
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial'] + plt.rcParams['font.sans-serif']
fontsize = 20

In [ ]:
ncols = 2
limit = 4

for exp_name, exp_data in experiment_neutron_data.items():
    print(exp_name)
    uncaught_pileup_df = exp_data["uncaught_pileup_df"]
    # pileup_peaks = data_dict["pileup_peaks"]

    nrows = ceil(min(uncaught_pileup_df.shape[1], limit) / ncols)

    fig, axs = plt.subplots(
        ncols=ncols,
        nrows=nrows,
        figsize=(7*ncols, 7*nrows),
        sharex=True,
        sharey=True
    )
    for ax_row in axs:
        ax_row[0].set_ylabel("Pulse height (ADC channel)", fontsize=fontsize)
    for ax in axs[-1]:
        ax.set_xlabel("Time (ns)", fontsize=fontsize)
    axs = axs.flatten()
    for ax in axs:
        ax.tick_params(labelsize=fontsize)
    for i, (signal_id, signal_data) in enumerate(uncaught_pileup_df.items()):
        if i >= limit:
            break
        neutron_x = signal_data.index.map(lambda x: int(x) * 2)
        neutron_y = signal_data.values
        axs[i].plot(neutron_x, neutron_y, label=signal_id)

In [ ]:
for exp_name, exp_data in experiment_neutron_data.items():
    print(exp_name)
    neutron_subset_signals = exp_data["subset_signals"]
    neutron_pileup_signals = exp_data["pileup_signals"]
    neutron_saturated_signals = exp_data["saturated_signals"]
    # neutron_max_triggers_signals = exp_data["max_trigger_signals"]
    neutron_clean_signals = exp_data["clean_signals"]

    fig, axs = plt.subplots(
        nrows=4,
        figsize=(7, 20)
    )
    fig.subplots_adjust(hspace=0.1)
    axs[-1].set_xlabel("Time (ns)", fontsize=fontsize)

    signal_dfs = [
        neutron_subset_signals,
        # neutron_pileup_signals,
        # neutron_saturated_signals,
        neutron_clean_signals
    ]
    for ax, signal_df in zip(axs, signal_dfs):
        pulse_countdown = 1000
        for row_idx, row_data in signal_df.iterrows():
            if pulse_countdown <= 0:
                break
            neutron_x = row_data.index.map(lambda x: int(x) * 2)
            neutron_y = row_data.values
            ax.plot(neutron_x, neutron_y)
            ax.set_ylabel("Pulse height (ADC channel)", fontsize=fontsize)
            ax.tick_params(labelsize=fontsize)
            pulse_countdown -= 1
    

In [ ]:
for exp_name, exp_data in experiment_neutron_data.items():
    print(exp_name)
    neutron_subset_signals = exp_data["subset_signals"]
    neutron_pileup_signals = exp_data["pileup_signals"]
    neutron_saturated_signals = exp_data["saturated_signals"]
    # neutron_max_triggers_signals = exp_data["max_trigger_signals"]
    neutron_clean_signals = exp_data["clean_signals"]

    # fig, axs = plt.subplots(
    #     nrows=4,
    #     figsize=(7, 20)
    # )
    fig, axs = plt.subplots(nrows=5, figsize=(7, 20), sharex=True, sharey=True)
    fig.subplots_adjust(hspace=0.1)

    # signal_dfs = [
    #     neutron_subset_signals,
    #     neutron_pileup_signals,
    #     neutron_saturated_signals,
    #     neutron_clean_signals
    # ]
    # for ax, signal_df in zip(axs, signal_dfs):
    #     pulse_countdown = 5
    #     for row_idx, row_data in signal_df.iterrows():
    #         if pulse_countdown <= 0:
    #             break
    #         neutron_x = row_data.index.map(lambda x: int(x) * 2)
    #         neutron_y = row_data.values
    #         ax.plot(neutron_x, neutron_y)
    #         pulse_countdown -= 1
    for i, (row_idx, row_data) in enumerate(neutron_clean_signals.iterrows()):
        if i >= len(axs):
            break
        neutron_x = row_data.index.map(lambda x: int(x) * 2)
        neutron_y = row_data.values
        axs[i].plot(neutron_x, neutron_y)
        axs[i].set_xlabel("Time (ns)", fontsize=fontsize)
        axs[i].set_ylabel("Pulse height (ADC channel)", fontsize=fontsize)
        axs[i].tick_params(labelsize=fontsize)
        

In [ ]:
input("Processing done, hit Enter to finish")
stop()